In [0]:
# ============================================================
# Silver Layer Configuration
# ============================================================

from pyspark.sql.functions import (
    col, explode, coalesce, lit, trim, broadcast
)
import time

CONFIG = {
    "bronze_db": "tvmaze.bronze",
    "silver_db": "tvmaze.silver",

    "bronze": {
        "shows": "tvmaze.bronze.tv_shows",
        "episodes": "tvmaze.bronze.episodes",
        "cast": "tvmaze.bronze.cast"
    },

    "silver": {
        "shows": "tvmaze.silver.silver_shows",
        "episodes": "tvmaze.silver.silver_episodes",
        "cast": "tvmaze.silver.silver_cast",
        "fact": "tvmaze.silver.fact_show_data"
    }
}

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CONFIG['silver_db']}")
spark.sql(f"USE {CONFIG['silver_db']}")

print("Silver config ready")

In [0]:
# ============================================================
# Read Bronze data
# ============================================================

bronze_shows    = spark.table(CONFIG["bronze"]["shows"])
bronze_episodes = spark.table(CONFIG["bronze"]["episodes"])
bronze_cast     = spark.table(CONFIG["bronze"]["cast"])

# display(bronze_shows.limit(3))
# display(bronze_episodes.limit(3))
# display(bronze_cast.limit(3))


In [0]:
# ============================================================
# Flatten silver shows data
# ============================================================
silver_shows = (
    bronze_shows
    .select(
        col("id").alias("show_id"),
        col("name").alias("show_name"),
        col("language"),
        col("premiered"),
        col("status"),
        col("runtime"),
        col("rating.average").alias("rating"),
        col("network.name").alias("network_name"),
        col("genres").alias("genres_array")
    )
    .withColumn("show_name", trim(coalesce(col("show_name"), lit("Unknown Show"))))
    .withColumn("language", trim(coalesce(col("language"), lit("Unknown"))))
)

# Normalize genres
silver_shows = (
    silver_shows
    .withColumn("genre", explode(coalesce(col("genres_array"), lit([]))))
    .drop("genres_array")
)

# display(silver_shows.limit(6))


In [0]:
# ============================================================
# Flatten silver episodes data
# ============================================================
silver_episodes = (
    bronze_episodes
    .select(
        col("id").alias("episode_id"),
        col("name").alias("episode_name"),
        col("season"),
        col("number").alias("episode_number"),
        col("airdate"),
        col("runtime"),
        col("show_id")
    )
    .withColumn("episode_name", trim(coalesce(col("episode_name"), lit("Unknown Episode"))))
    .withColumn("runtime", coalesce(col("runtime"), lit(0)))
)

# display(silver_episodes.limit(5))


In [0]:
# ============================================================
# Flatten silver cast data
# ============================================================

silver_cast = (
    bronze_cast
    .select(
        col("show_id"),
        col("person.id").alias("person_id"),
        col("person.name").alias("cast_name"),
        col("character.name").alias("character_name")
    )
    .withColumn("cast_name", trim(coalesce(col("cast_name"), lit("Unknown Cast"))))
    .withColumn("character_name", trim(coalesce(col("character_name"), lit("Unknown Character"))))
)

# display(silver_cast.limit(5))


In [0]:
# ============================================================
# Write silver tables
# ============================================================

def write_silver(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("mergeSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"Created Silver table: {table_name}")

write_silver(silver_shows, CONFIG["silver"]["shows"])   
write_silver(silver_episodes, CONFIG["silver"]["episodes"])
write_silver(silver_cast, CONFIG["silver"]["cast"])


In [0]:
# ============================================================
# Fact Table creation
# ============================================================

shows = spark.table(CONFIG["silver"]["shows"])
episodes = spark.table(CONFIG["silver"]["episodes"])
cast = spark.table(CONFIG["silver"]["cast"])

# Broadcast join on cast (small dimension table)
cast_b = broadcast(cast)

fact_df = (
    episodes.alias("e")
    .join(shows.alias("s"), "show_id", "inner")
    .join(cast_b.alias("c"), "show_id", "left")
    .select(
        col("s.show_id"),
        col("s.show_name"),
        col("s.language"),
        col("s.genre"),
        col("e.season"),
        col("e.episode_name"),
        col("e.airdate"),
        col("e.runtime"),
        col("c.cast_name"),
        col("c.character_name")
    )
)

fact_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("show_id") \
    .saveAsTable(CONFIG["silver"]["fact"])

print("fact_show_data table created")

In [0]:
# ============================================================
# Before Optimization Runtime
# ============================================================
start = time.time()

fact_df.count()

end = time.time()

print("Runtime:", end - start)  # 9.373462200164795

In [0]:
# ============================================================
# Optimize Delta Table
# ============================================================

spark.sql("""
OPTIMIZE tvmaze.silver.fact_show_data
ZORDER BY (genre, season)
""")

print("OPTIMIZE and ZORDER completed")

In [0]:
# ============================================================
# After Optimization Runtime
# ============================================================

start = time.time()

fact_df.count()

end = time.time()

print("Runtime:", end - start)  # 0.6226396560668945

In [0]:
from datetime import datetime

log = {
    "notebook": "Data_Transformations",
    "status": "Succeeded",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "notes": "Silver tables created"
}

print(log)
